In [ ]:
# --- locate the package and the data root, then run from the data root ---
# Layout after moving the code into a subfolder:
#   <DATA_ROOT>/                       <- chdir here; config paths resolve here
#       train/  embeddings/  outputs/  checkpoints/
#       multiomic-missing-data-pipeline/
#           transformer_pipeline/      <- the importable package
#           experiments/               <- this notebook
# Walk up from the cwd to find each marker independently, so it works no matter
# where the kernel was launched. Idempotent: safe to re-run.
import os, sys


def _find_up(start, marker):
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Could not locate '{marker}/' above {start!r}.")
        d = parent


# Package dir = the folder that *contains* `transformer_pipeline/`.
_PKG_PARENT = _find_up(os.getcwd(), "transformer_pipeline")
_PKG = os.path.join(_PKG_PARENT, "transformer_pipeline")
if _PKG not in sys.path:
    sys.path.insert(0, _PKG)

# Data root = the nearest folder that holds `train/` (the input CSVs).
_DATA_ROOT = _find_up(os.getcwd(), "train")
os.chdir(_DATA_ROOT)

print("cwd ->", os.getcwd())
print("pkg ->", _PKG)


# Stage 0 - Honest metric (per-feature r + select on val-MSE)

This verifies the Stage 0 change to the **pipeline** (not the ablation notebooks):

1. Every training epoch now prints **`per_feat_r`** = mean per-feature Pearson r
   on the validation set - the honest "did we predict individual features"
   number.
2. Model selection / early stopping now uses **`val_mse`** instead of Mantel.
   (Mantel is still printed each epoch, but only as a secondary report - the
   ridge probe showed it does not track imputation quality.)
   - The metric being selected on is marked with a `*` in the log
     (`val_mse=...*`).

**What to look for**
- The per-epoch lines contain `per_feat_r=...`.
- The selected (best) epoch is the one with the lowest `val_mse`.
- The best per-feature r should land near the *current* model level
  (~0.04 for A, ~0.01 for B) - Stage 0 changes the yardstick, not the model,
  so we do NOT expect a jump yet. The ridge ceiling (A ~0.12, B ~0.14) is the
  target for later stages.

Run this from the repo root (the folder containing `pipeline/`): Cell > Run All.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import pandas as pd
from pipeline.config import Config
from pipeline.pipeline import ImputationPipeline

cfg = Config()
print("selection_metric =", cfg.train.selection_metric, "(expect 'val_mse')")

# Keep it quick for a check. Mantel is only a secondary report now, so use few
# permutations to speed up the per-epoch evaluation.
cfg.train.max_epochs = 40
cfg.train.patience   = 40            # disable early stop so you see the full curve
cfg.eval.mantel_permutations = 99

### Train both models (watch the per-epoch `per_feat_r` and the `*` on `val_mse`)

In [ ]:
pipe = ImputationPipeline(cfg)
pipe.prepare_data()
trainer_a = pipe.train_model_a()
trainer_b = pipe.train_model_b()

### Summary: what was selected, and how it compares to the ridge ceiling

In [ ]:
def hist_df(trainer):
    return pd.DataFrame([{
        "epoch": r.epoch,
        "train_mse": round(r.train_loss, 5),
        "val_mse": round(r.val_mse, 5),
        "per_feat_r": (round(r.per_feature_r, 4) if r.per_feature_r is not None else None),
        "mantel_r": (round(r.mantel.statistic, 4) if r.mantel is not None else None),
    } for r in trainer.history.records])

dfa = hist_df(trainer_a)
dfb = hist_df(trainer_b)

def selected_row(trainer, df):
    e = trainer.history.best_epoch
    return df[df.epoch == e].iloc[0]

print("RIDGE CEILING (target for later stages):  A per_feat_r ~0.12   B per_feat_r ~0.14")
print()
ra, rb = selected_row(trainer_a, dfa), selected_row(trainer_b, dfb)
print(f"Model A  selected epoch {trainer_a.history.best_epoch:>2}  "
      f"val_mse={ra.val_mse}  per_feat_r={ra.per_feat_r}  mantel_r={ra.mantel_r}")
print(f"Model B  selected epoch {trainer_b.history.best_epoch:>2}  "
      f"val_mse={rb.val_mse}  per_feat_r={rb.per_feat_r}  mantel_r={rb.mantel_r}")
print()
print("Sanity check: the selected epoch should be the MIN val_mse epoch.")
print(f"  A  min-val_mse epoch = {int(dfa.loc[dfa.val_mse.idxmin(), 'epoch'])}")
print(f"  B  min-val_mse epoch = {int(dfb.loc[dfb.val_mse.idxmin(), 'epoch'])}")

In [ ]:
dfa.tail(12)

In [ ]:
dfb.tail(12)

### How to read this

- If `per_feat_r` appears on every line and the selected epoch = the min-`val_mse`
  epoch, Stage 0 is working.
- This stage is **measurement only** - per-feature r staying near 0.04 / 0.01 is
  expected. The point is that from now on we are selecting and judging on a
  metric that actually reflects imputation quality, so Stages 1-4 are
  interpretable.
- You can also run it from the terminal instead of this notebook:
  ```
  python run.py \
    --micro-emb embeddings/aligned/microbe_embeddings.csv \
    --metab-emb embeddings/aligned/metabolite_embeddings.csv \
    --max-epochs 40 --selection-metric val_mse --permutations 99
  ```